# Deep-Mechinterp Starter: AtomicDance Planner — v2 (reviewt & gefixt)

Aufbau wie gehabt: Stimulus-Suite → Aktivierungs-Hooks → Kontrastanalyse → lineare Probes → D3PM-Trajektorie → Kausal-Skelett. Das echte, selbst trainierte Planner-Modell wird **nur lesend** von Drive geladen; der Stimulus ist Musik (Faktoren aus den AIST++-Sequenznamen).

**Änderungen in v2 (nach Review):**

1. **Modell-Rekonstruktion gefixt (Abschnitt 5).** Die Auto-Discovery ist durch die verifizierte direkte Konstruktion ersetzt: `UniformD3PM(AtomicPlannerTransformer(num_atomic_classes=100, music_dim=35, max_seq_len=150))`. Im Quellcode gilt `num_classes = num_atomic_classes + 1` → Checkpoint-Embedding `[101, 512]` ⇒ **K=100**. `max_seq_len=150` ist zwingend (Klassen-Default 18000 ergäbe einen `pe`-Buffer `[18000, 1, 512]` → Size-Mismatch). Lokal gegen den Checkpoint-Abdruck bestätigt: 110 Tensoren, 19.121.965 Elemente, `strict=True` lädt.
   *Nebenbefund: `load_state_dict(strict=False)` wirft bei Size-Mismatch trotzdem `RuntimeError` — daran starb die alte K-Schleife, obwohl K=100 vermutlich schon erfolgreich geladen hatte.*
2. **Parameterdiskrepanz erklärt:** 19.121.965 (State-Dict) = 19.044.965 trainierbar + 76.800 (`pe [150, 1, 512]`) + 200 (`alphas` + `alpha_bars`). Kein Problem — Buffer vs. Parameter.
3. **Probe-Split gefixt:** `GroupShuffleSplit` gruppiert jetzt nach **Musik-ID** statt `base_seq`. Der Probe-Input ist ausschließlich Musik, und derselbe Song wird in AIST++ von mehreren Tänzern getanzt — sonst landet identischer Input in Train **und** Test (Leakage).
4. **Situations-Probe = Negativkontrolle:** sBM/sFM teilen dieselben Songs; erwartet ist Chance-Niveau. Deutlich mehr ⇒ Leakage-/Artefakt-Alarm.
5. **Trajektorie über mehrere Sequenzen gemittelt**; Chance- **und** Majority-Baseline im Probe-Plot; Baseline-Zuordnung über Facet-Titel (vorher konnte die Chance-Linie im falschen Facet landen).

Alle Ergebnisse (CSV/NPZ/PNG) landen dauerhaft unter `MyDrive/AtomicDance/mechinterp/`. Laufzeit: **T4 reicht völlig** (19-Mio-Parameter-Modell).

## 1) Runtime-Check

In [ ]:
import shutil, subprocess, sys, platform
print("Python:", sys.version.split()[0], "|", platform.platform())
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
else:
    print("Keine GPU aktiv — Analyse laeuft auch auf CPU, nur langsamer.")

## 2) Setup: Drive, Ordner, Seeds, Pakete

In [ ]:
%pip install -q einops scikit-learn seaborn

In [ ]:
from google.colab import drive
from pathlib import Path
import os, random
import numpy as np
import torch

drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DRIVE = Path('/content/drive/MyDrive/AtomicDance')
MI = DRIVE / 'mechinterp'
DATA_DIR, ACT_DIR, OUT_DIR = MI / 'data', MI / 'activations', MI / 'outputs'
for p in [DATA_DIR, ACT_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| device:', DEVICE)
print('Outputs →', MI)

## 3) Repo klonen & Testdaten aus dem Drive-Cache

Geklont wird **dein Fork** (`Erikiss/AtomicDance`, Stand: identisch mit `oceanflowlab/AtomicDance`, Commit `d790dbc`) — so bleibt der Lauf reproduzierbar, auch wenn sich das Original-Repo später ändert.

In [ ]:
%cd /content
!rm -rf /content/AtomicDance
!git clone -q https://github.com/Erikiss/AtomicDance.git
%cd /content/AtomicDance

In [ ]:
import glob, os, shutil, zipfile, tarfile

archives = sorted(glob.glob(str(DRIVE / 'dataset' / '*')))
assert archives, 'Kein Datensatz-Archiv im Drive-Cache (MyDrive/AtomicDance/dataset).'
src = archives[0]
dst = '/content/AtomicDance/data/atomic_aistpp'
if not os.path.exists(os.path.join(dst, 'manifest.json')):
    tmp = '/content/_atomic_extract'
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    (zipfile.ZipFile(src).extractall(tmp) if src.endswith('.zip')
     else tarfile.open(src).extractall(tmp))
    hit = next(r for r, d, f in os.walk(tmp) if 'manifest.json' in f)
    os.makedirs('/content/AtomicDance/data', exist_ok=True)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.move(hit, dst)
print('Datensatz bereit:', dst)

test_music = np.load(os.path.join(dst, 'test', 'music.npy'))
test_labels = np.load(os.path.join(dst, 'test', 'labels.npy'))
import json as _json
with open(os.path.join(dst, 'test', 'names.json')) as f:
    test_names = _json.load(f)
print('music:', test_music.shape, '| labels:', test_labels.shape, '| names:', len(test_names))
print('Beispielname:', test_names[0])

## 4) Checkpoint-Inspektion (nur lesend)

Neuesten Planner-Checkpoint von Drive laden und hineinschauen: Keys, Trainings-Args, Parameterzahl. **Erwartung:** 19.121.965 State-Dict-Elemente = 19.044.965 trainierbare Parameter (Trainingslog) + 77.000 Buffer (`pe [150, 1, 512]` = 76.800, `alphas`/`alpha_bars` = 200).

In [ ]:
ckpts = sorted((DRIVE / 'runs' / 'atomic_planner').glob('*.pt'), key=os.path.getmtime)
assert ckpts, 'Kein Planner-Checkpoint auf Drive gefunden.'
CKPT_PATH = ckpts[-1]
print('Checkpoint:', CKPT_PATH)

try:
    ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location='cpu')

print('Top-Level-Keys:', list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))

def find_state_dict(obj):
    if isinstance(obj, dict):
        for k in ['model', 'model_state', 'model_state_dict', 'state_dict', 'ema', 'ema_model']:
            if k in obj and isinstance(obj[k], dict):
                return obj[k], k
        if obj and all(torch.is_tensor(v) for v in obj.values()):
            return obj, '<root>'
    raise RuntimeError('State-Dict im Checkpoint nicht identifiziert — Keys oben pruefen.')

STATE, state_key = find_state_dict(ckpt)
STATE = { (k[7:] if k.startswith('module.') else k): v for k, v in STATE.items() }
n_params = sum(v.numel() for v in STATE.values() if torch.is_tensor(v))
print(f'State-Dict unter "{state_key}" | Tensoren: {len(STATE)} | Elemente: {n_params:,}')

CKPT_ARGS = {}
for k in ['args', 'config', 'hparams']:
    if isinstance(ckpt, dict) and k in ckpt:
        a = ckpt[k]
        CKPT_ARGS = dict(vars(a)) if hasattr(a, '__dict__') else dict(a)
        break
print('Trainings-Args:', CKPT_ARGS if CKPT_ARGS else '(keine im Checkpoint — nutze README-Defaults)')
print('\nErste State-Keys:', list(STATE.keys())[:8])

## 5) Planner rekonstruieren & Gewichte laden (verifiziert, v2)

Keine Discovery mehr nötig — `model/atomic_planner.py` definiert:

- `AtomicPlannerTransformer(num_atomic_classes, music_dim, latent_dim=512, num_layers=8, num_heads=8, ff_size=1024, dropout=0.1, max_seq_len=18000)` mit intern **`num_classes = num_atomic_classes + 1`**
- `UniformD3PM(model, num_steps=100, schedule='cosine')`

Die Konstruktion unten spiegelt exakt `train_atomic.py` und nutzt die im Checkpoint gespeicherten Trainings-Args (README-Defaults als Fallback). `schedule` ist egal — die Buffer werden vom Checkpoint überschrieben.

In [ ]:
import sys
import torch.nn as nn

sys.path.insert(0, '/content/AtomicDance')
from model.atomic_planner import AtomicPlannerTransformer, UniformD3PM

# Verifiziert: Checkpoint label_embedding [101,512] => num_atomic_classes=100.
# max_seq_len=150 zwingend (Default 18000 gaebe pe-Buffer [18000,1,512] -> Size-Mismatch).
PLANNER = UniformD3PM(
    AtomicPlannerTransformer(
        num_atomic_classes=int(CKPT_ARGS.get('num_classes', 100)),
        music_dim=int(CKPT_ARGS.get('music_dim', 35)),
        latent_dim=int(CKPT_ARGS.get('latent_dim', 512)),
        num_layers=int(CKPT_ARGS.get('num_layers', 8)),
        num_heads=int(CKPT_ARGS.get('num_heads', 8)),
        ff_size=int(CKPT_ARGS.get('ff_size', 1024)),
        dropout=float(CKPT_ARGS.get('dropout', 0.1)),
        max_seq_len=int(CKPT_ARGS.get('seq_len', 150)),
    ),
    num_steps=int(CKPT_ARGS.get('diffusion_steps', 100)),
)
PLANNER.load_state_dict(STATE, strict=True)
PLANNER.to(DEVICE).eval()

n_train = sum(p.numel() for p in PLANNER.parameters())
n_all = sum(v.numel() for v in PLANNER.state_dict().values())
print(f'✔ UniformD3PM(AtomicPlannerTransformer), K={PLANNER.num_classes - 1}')
print(f'  trainierbare Parameter: {n_train:,} (Soll: 19.044.965)')
print(f'  State-Dict inkl. Buffer: {n_all:,} (Soll: 19.121.965)')

## 6) Forward-Check

Die Signatur ist aus dem Repo bekannt: `forward(noisy_labels, music_features, timesteps, padding_mask=None)`. Standard-Analysezustand: `y` = uniformes Rauschen (fester Seed), `t = T_MAX − 1` — das Modell sieht dann effektiv nur die Musik, ideal für Musik-Probes.

In [ ]:
import inspect

CORE = PLANNER.model
print('Gehookter Kern:', type(CORE).__name__)
print('forward-Signatur:', inspect.signature(CORE.forward))

NUM_CLASSES = CORE.num_classes            # 101: Label 0 = Transition, 1-100 = Movements
SEQ_LEN = test_music.shape[1]             # 150
T_MAX = int(PLANNER.num_steps)            # 100

g = torch.Generator().manual_seed(SEED)
NOISE_Y = torch.randint(0, NUM_CLASSES, (1, SEQ_LEN), generator=g)

def core_forward(music_batch, y=None, t=None):
    B = music_batch.shape[0]
    y = (NOISE_Y.repeat(B, 1) if y is None else y).to(DEVICE)
    t = torch.full((B,), T_MAX - 1 if t is None else t, dtype=torch.long, device=DEVICE)
    m = music_batch.to(DEVICE).float()
    return CORE(y, m, t)  # forward(noisy_labels, music_features, timesteps, padding_mask=None)

with torch.no_grad():
    logits = core_forward(torch.tensor(test_music[:2]))
print('Logits-Shape:', tuple(logits.shape), '(erwartet: (2, 150, 101))')
assert logits.shape == (2, SEQ_LEN, NUM_CLASSES), 'Unerwartete Ausgabeform — bitte melden.'

## 7) Stimulus-Suite: Faktoren aus den Sequenznamen

Aus jedem Testfenster-Namen (`gBR_sBM_cAll_d04_mBR0_ch01_slice0`) parsen wir: **Genre** (10 Klassen), **Situation** (Basic/Advanced), **Tänzer**, **Musikstück** und die **Tempoklasse** (letzte Ziffer der Musik-ID; in der AIST-DB entspricht 0–5 aufsteigendem BPM von ca. 80–130).

In [ ]:
import pandas as pd
import re

def parse_name(name):
    m = {
        'genre': re.search(r'g([A-Z]{2})', name),
        'situation': re.search(r'_s([A-Z]{2})_', name),
        'dancer': re.search(r'_d(\d+)_', name),
        'music': re.search(r'_m([A-Z]{2}\d)', name),
        'choreo': re.search(r'_ch(\d+)', name),
    }
    row = {k: (v.group(1) if v else 'NA') for k, v in m.items()}
    row['tempo_class'] = int(row['music'][-1]) if row['music'] != 'NA' and row['music'][-1].isdigit() else -1
    row['base_seq'] = re.sub(r'(_slice.*|_win.*|_\d+)$', '', name)
    return row

meta = pd.DataFrame([{'idx': i, 'name': n, **parse_name(n)} for i, n in enumerate(test_names)])
parse_ok = (meta['genre'] != 'NA').mean()
print(f'Parse-Rate: {parse_ok:.1%} | Fenster: {len(meta)}')
print(meta['genre'].value_counts().to_dict())
print('Tempoklassen:', sorted(meta['tempo_class'].unique()))

N_STIM = min(512, len(meta))
stim = (meta.groupby('genre', group_keys=False)
            .apply(lambda d: d.sample(min(len(d), max(1, N_STIM // meta['genre'].nunique())), random_state=SEED))
            .reset_index(drop=True))
stim.to_csv(DATA_DIR / 'planner_stimulus_suite.csv', index=False)
print('Stimulus-Suite:', len(stim), '→', DATA_DIR / 'planner_stimulus_suite.csv')
stim.head()

## 8) Layer-Stack auswählen

Der Kern ist ein `nn.TransformerEncoder` mit 8 identischen Layern und **`batch_first=True`** — Layer-Outputs sind `[B, 150, 512]`. Wir hooken Anfang / Viertel / Mitte / Dreiviertel / Ende. (Die `enable_nested_tensor`-UserWarning kommt von `norm_first=True` und ist harmlos.)

In [ ]:
LAYERS = CORE.encoder.layers   # nn.ModuleList, 8x TransformerEncoderLayer (batch_first=True)
n_layers = len(LAYERS)
SELECTED_LAYERS = sorted(set([0, n_layers // 4, n_layers // 2, (3 * n_layers) // 4, n_layers - 1]))
print(f'Layer: {n_layers} | ausgewaehlt: {SELECTED_LAYERS} | Layer-Klasse: {type(LAYERS[0]).__name__}')

## 9) Activation-Recorder

Context-Manager mit Forward-Hooks. Musik ist kein kausaler Text — es gibt kein sinnvolles „letztes Token", deshalb **Mean-Pooling über die 150 Frames** (optional `pool='none'` für die volle `[150, d]`-Karte, z. B. für spätere Beat-Analysen).

In [ ]:
class ActivationRecorder:
    def __init__(self, layers, layer_indices, pool='mean'):
        self.layers, self.layer_indices, self.pool = layers, list(layer_indices), pool
        self.cache, self.handles = {}, []

    def _extract(self, output):
        h = output[0] if isinstance(output, (tuple, list)) else output
        if h.dim() == 3 and h.shape[0] == SEQ_LEN and h.shape[1] != SEQ_LEN:
            h = h.transpose(0, 1)  # [T,B,D] -> [B,T,D]; greift hier nie (batch_first=True)
        return h

    def _hook(self, idx):
        def fn(module, inputs, output):
            h = self._extract(output).detach().float()
            self.cache[idx] = (h.mean(dim=1) if self.pool == 'mean' else h).cpu()
        return fn

    def __enter__(self):
        self.handles = [self.layers[i].register_forward_hook(self._hook(i)) for i in self.layer_indices]
        return self

    def __exit__(self, *a):
        for h in self.handles: h.remove()

@torch.no_grad()
def capture(music_batch, layer_indices=None, pool='mean', y=None, t=None):
    layer_indices = SELECTED_LAYERS if layer_indices is None else layer_indices
    with ActivationRecorder(LAYERS, layer_indices, pool) as rec:
        core_forward(music_batch, y=y, t=t)
    return {i: rec.cache[i].numpy() for i in layer_indices}

test_acts = capture(torch.tensor(test_music[:2]))
{k: v.shape for k, v in test_acts.items()}

## 10) Kontrast-Analyse

Gruppen-Zentroiden statt einzelner Minimalpaare — robuster: **Genre** (gBR vs gHO), **Tempo** (Klasse 0 vs 5) und **Situation** (sBM vs sFM). Gemessen werden L2-Distanz und Cosine-Ähnlichkeit der mittleren Aktivierung je Layer.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def group_centroids(df_a, df_b, batch=32):
    def acts_for(df):
        sums = {i: 0 for i in SELECTED_LAYERS}; n = 0
        for s in range(0, len(df), batch):
            mb = torch.tensor(test_music[df['idx'].values[s:s+batch]])
            a = capture(mb)
            for i in SELECTED_LAYERS: sums[i] = sums[i] + a[i].sum(0)
            n += len(a[SELECTED_LAYERS[0]])
        return {i: sums[i] / n for i in SELECTED_LAYERS}
    return acts_for(df_a), acts_for(df_b)

def cosine(a, b, eps=1e-9):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

CONTRASTS = [
    ('genre', 'gBR', 'gHO', stim[stim.genre == 'BR'], stim[stim.genre == 'HO']),
    ('tempo', 'langsam(0)', 'schnell(5)', stim[stim.tempo_class == 0], stim[stim.tempo_class == 5]),
    ('situation', 'sBM', 'sFM', stim[stim.situation == 'BM'], stim[stim.situation == 'FM']),
]

rows = []
for axis, la, lb, da, db in CONTRASTS:
    if len(da) < 3 or len(db) < 3:
        print(f'{axis}: zu wenige Beispiele ({len(da)}/{len(db)}) — uebersprungen'); continue
    ca, cb = group_centroids(da, db)
    for i in SELECTED_LAYERS:
        d = ca[i] - cb[i]
        rows.append({'axis': axis, 'a': la, 'b': lb, 'layer': i,
                     'l2_diff': float(np.linalg.norm(d)),
                     'cosine': cosine(ca[i], cb[i]),
                     'n_a': len(da), 'n_b': len(db)})

contrast_df = pd.DataFrame(rows)
contrast_df.to_csv(OUT_DIR / 'planner_contrast_layer_metrics.csv', index=False)
display(contrast_df)

plt.figure(figsize=(8, 4))
sns.lineplot(data=contrast_df, x='layer', y='l2_diff', hue='axis', marker='o')
plt.title('Planner: Kontrast-Staerke pro Layer (Gruppen-Zentroiden)')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_contrast_l2_by_layer.png', dpi=160)
plt.show()

## 11) Aktivierungs-Datensatz für Probes (→ Drive)

In [ ]:
from tqdm.auto import tqdm

per_layer = {i: [] for i in SELECTED_LAYERS}
BATCH = 32
for s in tqdm(range(0, len(stim), BATCH)):
    mb = torch.tensor(test_music[stim['idx'].values[s:s+BATCH]])
    a = capture(mb)
    for i in SELECTED_LAYERS:
        per_layer[i].append(a[i])
ACTS = {i: np.concatenate(v).astype('float32') for i, v in per_layer.items()}

np.savez_compressed(ACT_DIR / 'planner_mean_activations.npz',
                    **{f'layer_{k}': v for k, v in ACTS.items()})
stim.to_csv(ACT_DIR / 'planner_activation_metadata.csv', index=False)
print({k: v.shape for k, v in ACTS.items()})
print('Gespeichert →', ACT_DIR)

## 12) Lineare Probes (Genre, Tempoklasse, Situation)

**v2-Fix:** Gruppiert wird nach **Musik-ID** (`music`-Spalte), nicht mehr nach `base_seq`. Der Modellinput ist bei `t = max` und fixem Noise-`y` **nur die Musik**, und derselbe Song (z. B. `mBR0`) kommt bei mehreren Tänzern/Choreos vor — `base_seq`-Gruppierung ließe identische Musikfeatures gleichzeitig in Train und Test (Leakage). Mit 6 Songs pro Genre wird der Split gröber (~1–2 ausgehaltene Songs pro Genre): verrauschter, aber ehrlich — und beantwortet die interessantere Frage, ob das Modell *Genre* kodiert oder nur *Song-Identität*.

**Situations-Probe (sBM vs. sFM) ist eine Negativkontrolle:** beide nutzen dieselben Songs — erwartet ist Chance-Niveau. Deutlich darüber ⇒ Artefakt/Leakage prüfen. Im Plot: gestrichelt rot = 1/K-Chance, gepunktet grau = Majority-Baseline im Test-Split.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

def run_probe(label_col):
    mask = stim[label_col].astype(str) != '-1'
    y = stim.loc[mask, label_col].astype(str).values
    # v2: Gruppierung nach Musik-ID — derselbe Song darf nie in Train UND Test liegen.
    groups = stim.loc[mask, 'music'].values
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
                  .split(np.zeros(len(y)), y, groups))
    _, counts = np.unique(y[te], return_counts=True)
    majority = counts.max() / counts.sum()
    recs = []
    for i in SELECTED_LAYERS:
        X = ACTS[i][mask.values]
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
        clf.fit(X[tr], y[tr])
        recs.append({'label': label_col, 'layer': i,
                     'accuracy': accuracy_score(y[te], clf.predict(X[te])),
                     'chance': 1.0 / len(np.unique(y)),
                     'majority': majority,
                     'n_test': len(te)})
    return pd.DataFrame(recs)

probe_df = pd.concat([run_probe(c) for c in ['genre', 'tempo_class', 'situation']], ignore_index=True)
probe_df.to_csv(OUT_DIR / 'planner_linear_probes.csv', index=False)
display(probe_df)

In [ ]:
g = sns.catplot(data=probe_df, x='layer', y='accuracy', col='label', kind='bar',
                color='#4c78a8', height=3.4, aspect=1.0)
for ax in g.axes.flat:
    lbl = ax.get_title().split(' = ')[-1]
    sub = probe_df[probe_df['label'] == lbl]
    ax.axhline(sub['chance'].iloc[0], ls='--', c='red', lw=1)
    ax.axhline(sub['majority'].iloc[0], ls=':', c='gray', lw=1)
    ax.set_ylim(0, 1)
g.figure.suptitle('Lineare Probes im Residual Stream des Planners', y=1.04)
g.figure.savefig(OUT_DIR / 'planner_linear_probes.png', dpi=160, bbox_inches='tight')
plt.show()

## 13) D3PM-Spezialität: Wann legen sich Entscheidungen fest?

Ein Pre-Hook auf dem Denoiser-Kern protokolliert bei jedem Reverse-Schritt den Label-Zustand `y_t` — der Sampler ruft den Kern genau **1× pro Schritt** mit `y_t` als `[B, 150]`-Long-Tensor auf (verifiziert in `UniformD3PM.sample`). Gemessen wird pro Schritt der Anteil der Frames, die schon dem finalen Plan entsprechen — getrennt für **Transitions** (Label 0) und **Movements** (1–100). Legt sich zuerst die Struktur fest oder zuerst der Inhalt?

**Interpretations-Hinweis:** `sample()` zieht in jedem Schritt *alle* Positionen neu aus `Categorical(logits)` — nichts ist je hart fixiert; die Kurve misst, wie früh die Logits scharf werden (effektive Kristallisation). Für reproduzierbare Rollouts: `deterministic=True`. **v2:** Mittel über mehrere Sequenzen (graue Linien = Einzelsequenzen).

In [ ]:
print('Sampler:', inspect.signature(PLANNER.sample))

traj_states = []

def pre_hook(module, args, kwargs):
    for v in list(args) + list(kwargs.values()):
        if torch.is_tensor(v) and v.dtype == torch.long and v.dim() == 2 and v.shape[-1] == SEQ_LEN:
            traj_states.append(v[0].detach().cpu().clone())
            break

def record_trajectory(idx, deterministic=False):
    traj_states.clear()
    music_1 = torch.tensor(test_music[idx:idx+1]).to(DEVICE).float()
    h = CORE.register_forward_pre_hook(pre_hook, with_kwargs=True)
    try:
        with torch.no_grad():
            plan = PLANNER.sample(music_1, deterministic=deterministic)
    finally:
        h.remove()
    final = plan[0].detach().cpu()
    rows = []
    for step, y in enumerate(traj_states):
        match = (y == final)
        rows.append({'idx': idx, 'step': step,
                     'match_all': match.float().mean().item(),
                     'match_transitions': match[final == 0].float().mean().item() if (final == 0).any() else np.nan,
                     'match_movements': match[final > 0].float().mean().item() if (final > 0).any() else np.nan})
    return pd.DataFrame(rows), final

In [ ]:
N_TRAJ = 16   # v2: ueber mehrere Sequenzen mitteln statt nur einer
trajs = []
for idx in stim['idx'].head(N_TRAJ):
    df_i, final = record_trajectory(int(idx))
    trajs.append(df_i)
traj_df = pd.concat(trajs, ignore_index=True)
traj_df.to_csv(OUT_DIR / 'planner_denoising_trajectory.csv', index=False)
print(f'{traj_df["idx"].nunique()} Sequenzen | {traj_df["step"].max() + 1} Reverse-Schritte')

mean_df = traj_df.groupby('step').mean(numeric_only=True).reset_index()
plt.figure(figsize=(8, 4))
for _, sub in traj_df.groupby('idx'):
    plt.plot(sub['step'], sub['match_all'], color='gray', alpha=0.15, lw=0.7)
for col, lbl in [('match_all', 'alle Frames'), ('match_transitions', 'Transitions (Label 0)'),
                 ('match_movements', 'Movements (1-100)')]:
    plt.plot(mean_df['step'], mean_df[col], label=lbl, lw=2)
plt.xlabel('Reverse-Diffusionsschritt'); plt.ylabel('Anteil = finaler Plan')
plt.title(f'Entscheidungs-Kristallisation im D3PM (Mittel ueber {N_TRAJ} Sequenzen)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_denoising_trajectory.png', dpi=160)
plt.show()

## 14) Kausal-Skelett: Music-Swap & Aktivierungs-Patching

(a) **Music-Swap** als Verhaltensbaseline: Plan mit Musik A vs. Musik B (anderes Genre) — wie stark ändern sich Transition-Anteil und Movement-Verteilung? (b) **Aktivierungs-Patch** (Skelett, nach den Probes mit dem besten Layer füllen): Layer-Outputs sind `[B, 150, 512]` (`batch_first=True`), `dim=1` ist also die **Zeitachse** — der Mean-Shift im Skelett ist dimensionsrichtig.

In [ ]:
def plan_stats(music_tensor, deterministic=False):
    with torch.no_grad():
        p = PLANNER.sample(music_tensor.to(DEVICE).float(), deterministic=deterministic)
    p = p[0].detach().cpu()
    segs = int((p[1:] != p[:-1]).sum()) + 1
    return {'transition_frac': float((p == 0).float().mean()),
            'n_segments': segs,
            'top_moves': torch.bincount(p[p > 0], minlength=NUM_CLASSES).topk(3).indices.tolist()}

a_row = stim[stim.genre == 'BR'].iloc[0]
b_row = stim[stim.genre == 'HO'].iloc[0]
m_a = torch.tensor(test_music[int(a_row['idx']):int(a_row['idx'])+1])
m_b = torch.tensor(test_music[int(b_row['idx']):int(b_row['idx'])+1])
print('Musik A (gBR):', plan_stats(m_a))
print('Musik B (gHO):', plan_stats(m_b))

In [ ]:
# --- Aktivierungs-Patch-Skelett (nach den Probes mit dem besten Layer fuellen) ---
# Layer-Outputs sind [B, 150, 512] (batch_first=True) -> dim=1 ist die Zeitachse.
# BEST_LAYER = 4
# source_centroid = torch.tensor(ACTS[BEST_LAYER][(stim.genre == 'BR').values].mean(0))
# def patch_hook(module, inputs, output):
#     h = output[0] if isinstance(output, (tuple, list)) else output
#     h = h + (source_centroid.to(h.device, h.dtype) - h.mean(dim=1, keepdim=True))
#     return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
# handle = LAYERS[BEST_LAYER].register_forward_hook(patch_hook)
# try:
#     print('gepatcht:', plan_stats(m_b))
# finally:
#     handle.remove()

## Ausblick

- **Beat-Heads:** mit `pool='none'` Frame-Aktivierungen `[150, d]` gegen die Beat-Dimensionen der 35 Musikfeatures korrelieren.
- **Head-Ablationen mit Verhaltens-Readout:** einzelne Attention-Heads nullen und den Effekt via `plan_stats` bzw. der vollen Eval (BAS) messen.
- **Music-Swap systematisch:** volle Genre×Genre-Matrix statt eines Einzelpaars; dazu `deterministic=True` für stabile Vergleiche.
- **Noise-Robustheit der Probes:** Aktivierungen über 4–8 verschiedene `NOISE_Y`-Seeds mitteln.

Alle Artefakte liegen unter `MyDrive/AtomicDance/mechinterp/` und überleben die Session.